In [110]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn import tree, metrics

In [111]:
historical = pd.read_csv('kep-historical.csv', header=None)
historical.head(n=5)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,70,M,109,145,34,1,1,O,71,b,a,68,M,0,B,81,ab,a,8
1,62,F,83,200,31,1,0,O,62,NaN,a,42,F,0,O,57,a,ab,4
2,73,M,120,160,29,0,1,A,66,ab,ab,40,F,1,A,62,ab,a,4
3,19,F,98,190,33,0,0,O,48,a,a,75,F,0,O,49,ab,b,11
4,41,F,122,100,28,1,0,B,35,ab,b,56,F,0,B,72,ab,b,15


In [112]:
don_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(historical[[1]])
don_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(historical[[7]])
don_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[9]])
don_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[10]])
pat_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(historical[[12]])
pat_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(historical[[14]])
pat_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[16]])
pat_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(historical[[17]])


In [113]:
donN = historical[[0,2,3,4,5,6,8]].values # numerical features
patN = historical[[11,13,15]].values # numerical features
X = np.concatenate((don_genders, don_hlaB, don_hlaDR, donN, pat_genders, pat_hlaB, pat_hlaDR, patN), axis=1)
y = historical[[18]].values
model = tree.DecisionTreeRegressor(random_state=0, max_depth=4)
model = model.fit(X, y)

In [114]:
ypred = model.predict(X)
print(f"compute errors using the training set -- *** not meaningful, just out of curiosity ***")
print(f"maximum error: {metrics.max_error(y, ypred)}")
print(f"mean absolute error: {metrics.mean_absolute_error(y, ypred)}")
print(f"mean squared error in the training set: {metrics.mean_squared_error(y, ypred)}")


compute errors using the training set -- *** not meaningful, just out of curiosity ***
maximum error: 18.815344603381014
mean absolute error: 4.134172319283718
mean squared error in the training set: 28.123169917141354


In [18]:
pool = pd.read_csv('kep-pool.csv', header=None)
don_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(pool[[2]])
don_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(pool[[8]])
don_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[10]])
don_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[11]])
pat_genders = OneHotEncoder(categories=[['F','M']], sparse_output=False).fit_transform(pool[[13]])
pat_btypes = OneHotEncoder(categories=[['A', 'B', 'AB', 'O', np.nan]], sparse_output=False).fit_transform(pool[[15]])
pat_hlaB = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[17]])
pat_hlaDR = OneHotEncoder(categories=[['a', 'b', 'ab', np.nan]], sparse_output=False).fit_transform(pool[[18]])
Xdon = np.concatenate((don_genders, don_hlaB, don_hlaDR, pool[[1,3,4,5,6,7,9]].values), axis=1)
Xpat = np.concatenate((pat_genders, pat_hlaB, pat_hlaDR, pool[[12,14,16]].values), axis=1)


In [60]:
pool.head(n=5)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1,56,F,79,145,31,1,0,A,82,b,ab,21,F,1,AB,81,b,b
1,2,39,M,85,100,30,0,0,O,95,a,ab,75,F,1,B,60,a,a
2,3,48,F,89,155,18,1,0,A,67,ab,ab,61,F,0,B,59,a,ab
3,4,35,F,129,165,34,0,0,A,58,b,a,48,M,0,B,104,a,b
4,5,72,M,72,200,33,0,0,O,55,NaN,ab,60,F,0,O,50,NaN,NaN


In [19]:
don_btype = pool[[8]].values.ravel()
pat_btype = pool[[15]].values.ravel()
n_pairs = len(pool)
for i in range(n_pairs):
    X = [list(Xdon[i]) + list(Xpat[i])]
    survival = model.predict(X)[0] # predicted survival time


In [118]:
threshold = 5 # patient survival time must be >= 5
compat = {
"O":["O"],
"A":["O","A"],
"B":["O", "B"],
"AB":["O", "A", "B", "AB"],
}
adj = {}
surv = {}
for j in range(n_pairs): # index for donor's pair
    adj[j] = [] # start with an empty list for each donor
    for i in range(n_pairs): # index for patient's pair
        if don_btype[j] in compat[pat_btype[i]]: # 'j' is blood-type compatible with 'i'
            X = [list(Xdon[j]) + list(Xpat[i])]
            survival = round(model.predict(X)[0],0) # predicted survival time
            if survival >= threshold: #
                adj[j].append(i)
                surv[j,i] = survival


In [ ]:
def trim_cycles_rationality(cycles:set,surv:dict):
    
    weights = []
    to_remove = []

    for cycle in cycles:
        cycle_len = len(cycle)
        weights.append(0)
        for i in range(cycle_len):
            patient_loc = 0
            if i+1 < cycle_len:
                patient_loc = int(cycle[i+1])
                weight = surv[(cycle[i],cycle[i+1])]
            else:
                patient_loc = int(cycle[0])
                weight = surv[(cycle[i],cycle[0])]
                       
            if cycle_len > 1 and (patient_loc,patient_loc) in surv.keys() and int(surv[(patient_loc,patient_loc)]) >= weight: #pode ser > ou >= - SE FOR >= ENTÃO VAMOS REMOVER OS CICLOS DE 1 TAMBÉM
                weights.pop()
                to_remove.append(cycle)
                break 
            weights[-1] += weight
            
    for i in range(len(to_remove)):
        cycles.remove(to_remove[i])
     
    return cycles, weights

In [40]:
import time
from amplpy import AMPL, Environment, DataFrame

In [122]:

def compute_cycle_weight(cycle, surv):
    total = 0
    for i in range(len(cycle)):
        a = cycle[i]
        b = cycle[(i + 1) % len(cycle)]
        total += surv[(a, b)]
    return total

def normalize(cycle):
    cmin = min(cycle)
    while cycle[0] != cmin:
        v = cycle.pop(0)
        cycle.append(v)


def all_cycles(cycles, path, node, tovisit, adj,K):
    for i in adj[node]:
        #if i == node:
        #    cycle = [node]
        #    cycles.add(tuple(cycle))
        if i in path:
            j = path.index(i)
            cycle = path[j:]+[node]
            normalize(cycle)
            cycles.add(tuple(cycle))

        if i in tovisit:
            if K-1 > 0:
                all_cycles(cycles,path+[node],i,tovisit-set([i]),adj,K-1)
    return cycles


def get_all_cycles(adj,K):
    tovisit = set(adj.keys())
    visited = set()
    cycles = set()
    for i in tovisit:
        tmpvisit = set(tovisit)
        tmpvisit.remove(i)
        first = i
        all_cycles(cycles,[],first,tmpvisit,adj,K)
    return cycles

In [94]:
def kep_cycle(adj,cycles, weights):
    ampl = AMPL()
    ampl.option['solver'] = 'gurobi'
    ampl.read("KEP_MAX.mod")
    ampl.set['V'] = list(adj)   # vertices are keys in 'adj'
    ampl.param['m'] = len(cycles)
    ampl.param['w'] = {i: weights[i] for i in range(len(weights))}
    for i,c in enumerate(cycles):
        ampl.set['CYCLE'][i] = list(c)
   
    ampl.solve()
    assert ampl.getValue('solve_result') == "solved"

    z = ampl.obj['z']
    x = ampl.var['x']
    sol = []
    for i,c in enumerate(cycles):
        if x[i].value() > 0.5:
            sol.append(c)
    return z.value(), sol

In [97]:
cycles = get_all_cycles(adj,3)
print("Cycles: ", len(cycles))
weights = [compute_cycle_weight(c, surv) for c in cycles]
z, sol = kep_cycle(adj, cycles, weights)
print("solution:", z, sol)

Cycles:  58971
Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 1146
24524 simplex iterations
1 branching node
solution: 1146.0 [(41, 70, 92), (27, 35, 52), (33,), (29, 85, 58), (37,), (94,), (21, 31, 56), (46, 47), (10, 49, 80), (48, 60, 72), (54,), (20, 98, 99), (34, 44, 57), (11, 30, 18), (19, 26, 95), (4, 38, 24), (36, 84, 90), (62, 76, 79), (1, 8, 97), (63,), (67, 83, 78), (32, 68, 50), (2, 77, 69), (17, 28, 74), (7, 15, 93), (64,), (23, 73, 43), (42, 81), (5, 91, 39), (6, 12, 88), (25, 59, 65), (3, 61, 22), (55, 87, 96), (89,), (75, 82, 86), (14, 40, 16), (13, 71, 66)]


Rationality Constraint

In [98]:
cycles = get_all_cycles(adj,3)
print("Length before trimming: ", len(cycles))
cycles, weights = trim_cycles_rationality(cycles,surv)
print("Length after trimming: ", len(cycles))
z, sol = kep_cycle(adj, cycles, weights)
print("solution:", z, sol)

Length before trimming:  58971
Length after trimming:  2943
Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 1068
243 simplex iterations
1 branching node
solution: 1068.0 [(13, 21, 44), (49,), (6, 77, 71), (16, 20), (29,), (30, 31, 85), (33,), (19, 23, 48), (4, 86, 72), (90,), (0, 97, 55), (1,), (82,), (58,), (5,), (25, 91, 74), (62,), (17, 57, 83), (38,), (10, 75, 78), (87,), (8, 81, 94), (67,), (69, 73), (47, 92, 54), (3, 46, 40), (7, 35, 79), (63,), (18, 80), (39,), (96,), (43,), (27,), (88,), (36, 84, 95), (60,), (64,), (11,), (68,), (56,), (32,), (14, 22, 99), (93,), (37, 50, 65), (24,), (41, 76), (28,), (89,), (2, 12), (61,), (98,)]


In [ ]:
# compat = {
#     "O": ["O"],
#     "A": ["O", "A"],
#     "B": ["O", "B"],
#     "AB": ["O", "A", "B", "AB"],
# }

# threshold = 5
# adj = {}
# surv = {}

# # First, precompute self-pair survival times
# self_surv = {}
# for i in range(n_pairs):
#     if don_btype[i] in compat[pat_btype[i]]:
#         X_self = [list(Xdon[i]) + list(Xpat[i])]
#         self_surv[i] = round(model.predict(X_self)[0])
#     else:
#         self_surv[i] = 0

# # Now build the adjacency list with both conditions
# for j in range(n_pairs):  # donor index
#     adj[j] = []
#     for i in range(n_pairs):  # patient index
#         if don_btype[j] in compat[pat_btype[i]]:
#             X = [list(Xdon[j]) + list(Xpat[i])]
#             survival = round(model.predict(X)[0], 0)
#             if survival >= threshold and survival >= self_surv[i]:  # Compare with self-match
#                 adj[j].append(i)
#                 surv[(j, i)] = survival

In [ ]:
# cycles = get_all_cycles(adj,3)
# print("Cycles: ", len(cycles))
# weights = [compute_cycle_weight(c, surv) for c in cycles]
# z, sol = kep_cycle(adj, cycles, weights)
# print("solution:", z, sol)

Cycles:  13648
Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 1122
650 simplex iterations
1 branching node
solution: 1122.0 [(8, 32), (12, 67, 18), (38,), (93,), (3, 43, 35), (47, 95, 57), (40, 99, 88), (30, 31, 85), (2, 36), (10, 80, 39), (24,), (25, 79, 96), (94,), (16, 76), (52, 75, 98), (21, 91, 92), (0, 97, 55), (14, 22, 62), (23, 48, 29), (5,), (60,), (41, 46, 83), (7, 34, 59), (13, 17, 90), (11,), (68,), (37, 56), (6, 49, 72), (4, 50, 65), (27, 64, 44), (1, 73, 20), (69, 82, 81), (63,), (54,), (28, 33), (78, 84, 87), (19, 26), (58, 74, 86), (61, 89, 70), (15, 77, 71)]


Decision Tree Classifier

In [115]:
y_train_classifier = (historical[[18]].values >= 5).astype(int)
model_clf = tree.DecisionTreeClassifier(random_state=0, max_depth=4)
model_clf = model_clf.fit(X, y_train_classifier)

In [116]:
ypred = model_clf.predict(X)
print(f"compute errors using the training set -- *** not meaningful, just out of curiosity ***")
print(f"maximum error: {metrics.max_error(y, ypred)}")
print(f"mean absolute error: {metrics.mean_absolute_error(y, ypred)}")
print(f"mean squared error in the training set: {metrics.mean_squared_error(y, ypred)}")

compute errors using the training set -- *** not meaningful, just out of curiosity ***
maximum error: 35
mean absolute error: 9.9102
mean squared error in the training set: 151.6694


In [123]:
adj = {}
surv = {}

for j in range(n_pairs): # index for donor's pair
    adj[j] = [] # start with an empty list for each donor
    for i in range(n_pairs): # index for patient's pair
        if don_btype[j] in compat[pat_btype[i]]: # 'j' is blood-type compatible with 'i'
            X = [list(Xdon[j]) + list(Xpat[i])]
            prob = max(model_clf.predict_proba(X)[0])
            res = model_clf.predict(X)[0]
            survival = round(model.predict(X)[0],0)
            if prob >= .75 and res == 1:
                adj[j].append(i)
                surv[j,i] = survival

In [124]:
cycles = get_all_cycles(adj,3)
print("Cycles: ", len(cycles))
weights = [compute_cycle_weight(c, surv) for c in cycles]
z, sol = kep_cycle(adj, cycles, weights)
print("solution:", z, sol)

Cycles:  15168
Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 859
5708 simplex iterations
1 branching node
solution: 859.0 [(33, 60), (82, 85, 97), (5, 99, 73), (15, 80, 98), (19, 25, 79), (22, 91, 29), (4, 86, 81), (3, 65, 40), (13, 56, 42), (12, 78, 44), (11, 49), (47, 77), (48, 93, 72), (24, 37), (39, 55), (23, 94, 89), (8, 75, 16), (41, 64, 83), (38, 68), (6, 28, 32), (27, 50), (17, 36, 63)]


In [125]:
cycles = get_all_cycles(adj,3)
print("Length before trimming: ", len(cycles))
cycles, weights = trim_cycles_rationality(cycles,surv)
print("Length after trimming: ", len(cycles))
z, sol = kep_cycle(adj, cycles, weights)
print("solution:", z, sol)

Length before trimming:  15168
Length after trimming:  920
Gurobi 12.0.1:Gurobi 12.0.1: optimal solution; objective 561
193 simplex iterations
1 branching node
solution: 561.0 [(4, 83, 41), (13, 25, 44), (23, 85, 65), (37, 50, 72), (8, 81, 56), (40, 73), (39, 79, 75), (16, 22, 63), (19, 94, 48), (49, 77, 86), (36, 91, 93), (17, 97, 78), (3, 11, 12), (47, 80, 99)]
